# 🤖 E-Commerce AI Sales & Profitability Analysis
### Executive Analytical Diagnostic, RFM Customer Modeling & Loss Decomposition
**Dataset Scope:** 500 Orders | 1,500 Line Items | Pan-India FY 2018

---

## 1. Project Objectives
1. **Audit Financial Data Quality:** Reconcile gross revenue, net margin, and correct the legacy **121K AOV bug**.
2. **Decompose Margin Leakage:** Identify why **35.3% of items** operated at a net loss, destroying **₹38,079** in profit.
3. **Perform Customer RFM Segmentation:** Uncover high-margin champions versus high-volume unprofitable buyers (the *Harivansh Paradox*).
4. **Prescribe Strategic Action Items:** Formulate automated pricing floors and COD risk mitigation.

In [1]:
import csv
from collections import defaultdict
from datetime import datetime

# Load Orders and Details data
with open('dataset/Orders.csv', 'r', encoding='utf-8') as f:
    orders = list(csv.DictReader(f))

with open('dataset/Details.csv', 'r', encoding='utf-8') as f:
    details = list(csv.DictReader(f))

print(f'Loaded {len(orders)} orders and {len(details)} item details.')


## 2. Executive KPI Calculation & The AOV Audit
In the legacy Power BI dashboard, the AOV card displayed **121K** due to an erroneous `SUM()` aggregation of row-level price ratios.
Below is the verified mathematical calculation:

In [2]:
total_revenue = sum(float(d['Amount']) for d in details)
total_profit = sum(float(d['Profit']) for d in details)
total_quantity = sum(int(d['Quantity']) for d in details)
distinct_orders = len(set(d['Order ID'] for d in details))

true_aov = total_revenue / distinct_orders
avg_item_price = total_revenue / total_quantity
net_margin = (total_profit / total_revenue) * 100

print(f'Total Revenue   : ₹{total_revenue:,.2f}')
print(f'Net Profit      : ₹{total_profit:,.2f} ({net_margin:.2f}% Margin)')
print(f'True AOV        : ₹{true_aov:.2f} (Corrected from 121K bug)')
print(f'Avg Item Price  : ₹{avg_item_price:.2f}')


## 3. Margin Leakage Diagnostic (The ₹38K Bleed)
Here we quantify the difference between positive gain transactions and loss-making line items:

In [3]:
profitable = [d for d in details if float(d['Profit']) >= 0]
loss_items = [d for d in details if float(d['Profit']) < 0]

gross_gains = sum(float(d['Profit']) for d in profitable)
gross_losses = sum(float(d['Profit']) for d in loss_items)

print(f'Profitable Items: {len(profitable)} ({len(profitable)/len(details)*100:.1f}%) -> Generated +₹{gross_gains:,.2f}')
print(f'Loss Items      : {len(loss_items)} ({len(loss_items)/len(details)*100:.1f}%) -> Destroyed -₹{abs(gross_losses):,.2f}')
print(f'\nKey Finding: Eliminating loss items doubles net profit from ₹{total_profit:,.2f} to ₹{gross_gains:,.2f} (+103% gain)!')


## 4. Sub-Category Profitability Rankings
Exposing the money-makers versus the cash drains:

In [4]:
subcat_summary = defaultdict(lambda: {'sales': 0.0, 'profit': 0.0, 'losses': 0, 'count': 0})
for d in details:
    s = d['Sub-Category']
    amt = float(d['Amount'])
    prof = float(d['Profit'])
    subcat_summary[s]['sales'] += amt
    subcat_summary[s]['profit'] += prof
    subcat_summary[s]['count'] += 1
    if prof < 0:
        subcat_summary[s]['losses'] += 1

sorted_subs = sorted(subcat_summary.items(), key=lambda x: x[1]['profit'], reverse=True)
print(f"{'Sub-Category':<18} | {'Revenue':<11} | {'Net Profit':<11} | {'Margin %':<8} | {'Loss Items':<10}")
print('-' * 70)
for sub, v in sorted_subs:
    margin = (v['profit'] / v['sales'] * 100) if v['sales'] else 0
    print(f"{sub:<18} | ₹{v['sales']:>9,.0f} | ₹{v['profit']:>9,.0f} | {margin:>7.1f}% | {v['losses']}/{v['count']}")


## 5. Customer RFM Segmentation & The Harivansh Paradox
Demonstrating why optimizing exclusively for gross sales leads to rewarding loss-making accounts:

In [5]:
order_map = {o['Order ID']: o for o in orders}
cust_stats = defaultdict(lambda: {'sales': 0.0, 'profit': 0.0, 'orders': set()})

for d in details:
    oid = d['Order ID']
    cname = order_map[oid]['CustomerName']
    cust_stats[cname]['sales'] += float(d['Amount'])
    cust_stats[cname]['profit'] += float(d['Profit'])
    cust_stats[cname]['orders'].add(oid)

print('Top 5 Spenders (Gross Sales):')
for c, v in sorted(cust_stats.items(), key=lambda x: x[1]['sales'], reverse=True)[:5]:
    print(f"  {c:<12}: Sales=₹{v['sales']:<7,.0f} | Net Profit=₹{v['profit']:>6,.0f} | Orders={len(v['orders'])}")

print('\nTop 5 Profit Contributors (Net Value):')
for c, v in sorted(cust_stats.items(), key=lambda x: x[1]['profit'], reverse=True)[:5]:
    print(f"  {c:<12}: Net Profit=₹{v['profit']:<7,.0f} | Sales=₹{v['sales']:>7,.0f} | Margin={v['profit']/v['sales']*100:.1f}%")


## 6. Executive Strategic Recommendations
1. **SKU Price Floors:** Cap discounting on *Electronic Games* (-₹644) and *Furnishings* (-₹806).
2. **COD Prepayment Nudge:** Implement a 2.5% instant discount for UPI to reduce the 35.4% COD exposure and mitigate courier returns.
3. **Mumbai Margin Recovery:** Rebalance product mix in Mumbai (1.4% margin) towards high-margin categories (Printers, Bookcases) mirroring Pune (14.1% margin).
4. **Automated 30-Day Retention Funnel:** Engage the 68.2% single-purchase customer segment.